# StandUp4AI Evaluation: F0 Prosody vs Baseline

**Goal:** Evaluate if our F0/spectral features beat StandUp4AI's F1=0.51 on their benchmark.

**Data:** 30 videos (we have audio+labels for these).

**Baseline to beat:** F1=0.51 (StandUp4AI text-based)

In [ ]:
# Setup
import os, json, subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/standup4ai')
AUDIO_DIR = BASE / 'audio'
LABELS_DIR = BASE / 'labels'

# Check what we have
audio_files = list(AUDIO_DIR.glob('*.m4a'))
label_files = list(LABELS_DIR.glob('*.csv'))
print(f'Audio files: {len(audio_files)}')
print(f'Label files: {len(label_files)}')

# Get overlapping video IDs
audio_vids = {f.stem for f in audio_files}
label_vids = {f.stem for f in label_files}
overlap = audio_vids & label_vids
print(f'Have both: {len(overlap)} videos')

# Install deps
subprocess.run(['pip', 'install', '-q', 'librosa', 'pandas', 'numpy', 'scikit-learn'], check=True)
import librosa, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load labels for overlapping videos
def load_laughter_labels(csv_path):
    """Load laughter intervals from CSV."""
    df = pd.read_csv(csv_path)
    df['label_bin'] = (df['label'] == 'risa').astype(int)
    return df

# Extract prosody features for a segment
def extract_features(audio_path, t0, t1, sr=22050):
    try:
        dur = min(t1 - t0, 10.0)
        if dur < 0.1:
            return None
        y, sr = librosa.load(audio_path, sr=sr, offset=t0, duration=dur)
        if len(y) < sr * 0.1:
            return None
        
        # F0 via pyin (fundamental frequency)
        try:
            f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=80, fmax=500, sr=sr)
            f0 = np.nan_to_num(f0, nan=0)
            f0_mean = np.mean(f0)
            f0_std = np.std(f0)
            voiced_rate = np.mean(voiced_flag)
        except:
            f0_mean, f0_std, voiced_rate = 0, 0, 0
        
        # Spectral features
        hop = 512
        rms = librosa.feature.rms(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        flat = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop, n_mfcc=13)
        
        return np.array([
            f0_mean, f0_std, voiced_rate,
            np.mean(rms), np.std(rms), np.max(rms),
            np.mean(zcr), np.std(zcr),
            np.mean(cent), np.std(cent),
            np.mean(bw), np.std(bw),
            np.mean(flat), np.std(flat),
            np.mean(mfcc[1]), np.std(mfcc[1]),
            np.mean(mfcc[2]), np.std(mfcc[2]),
            np.mean(mfcc[3]), np.std(mfcc[3]),
            len(y)/sr,
        ])
    except:
        return None

# Extract features for all overlapping videos
print('Extracting features...')
X_all, y_all, vids_all = [], [], []

for vid in sorted(overlap):
    audio_path = AUDIO_DIR / f'{vid}.m4a'
    label_path = LABELS_DIR / f'{vid}.csv'
    
    if not audio_path.exists() or not label_path.exists():
        continue
    
    labels = load_laughter_labels(label_path)
    for _, seg in labels.iterrows():
        feat = extract_features(str(audio_path), float(seg['t0']), float(seg['t1']))
        if feat is not None:
            X_all.append(feat)
            y_all.append(int(seg['label_bin']))
            vids_all.append(vid)

X = np.array(X_all)
y = np.array(y_all)
videos = np.array(vids_all)

print(f'\n=== DATASET ===')
print(f'Samples: {len(y)}')
print(f'Positive: {y.sum()} ({y.mean():.1%})')
print(f'Videos: {len(set(videos))}')
print(f'Features: {X.shape[1]}-dim')

In [ ]:
# Video-level cross-validation
print('\n=== VIDEO-LEVEL CV ===')

models = {
    'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1),
    'XGBoost': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
}

gkf = GroupKFold(n_splits=min(5, len(set(videos))))
results = {}

for name, model in models.items():
    f1s, precs, recs = [], [], []
    for tr, te in gkf.split(X, y, videos):
        if len(set(y[te])) < 2:
            continue
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        model.fit(Xtr, y[tr])
        pred = model.predict(Xte)
        f1s.append(f1_score(y[te], pred, zero_division=0))
        precs.append(precision_score(y[te], pred, zero_division=0))
        recs.append(recall_score(y[te], pred, zero_division=0))
    
    mean_f1 = np.mean(f1s) if f1s else 0
    std_f1 = np.std(f1s) if f1s else 0
    results[name] = {'f1': mean_f1, 'std': std_f1, 'folds': f1s}
    print(f'{name:12s} F1={mean_f1:.4f} ± {std_f1:.4f}')

print(f'\n{"="*50}')
print(f'StandUp4AI baseline: F1=0.51')
for name, r in sorted(results.items(), key=lambda x: -x[1]['f1']):
    beat = '🏆 BEATS BASELINE' if r['f1'] > 0.51 else ''
    print(f'{name:12s} F1={r["f1"]:.4f} ± {r["std"]:.4f} {beat}')
print(f'{"="*50}')

In [ ]:
# Feature importance
print('\n=== FEATURE IMPORTANCE ===')

sc = StandardScaler()
Xs = sc.fit_transform(X)
lr = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.1)
lr.fit(Xs, y)

feature_names = ['f0_mean', 'f0_std', 'voiced_rate',
                'rms_mean', 'rms_std', 'rms_max',
                'zcr_mean', 'zcr_std',
                'cent_mean', 'cent_std',
                'bw_mean', 'bw_std',
                'flat_mean', 'flat_std',
                'mfcc2_mean', 'mfcc2_std',
                'mfcc3_mean', 'mfcc3_std',
                'mfcc4_mean', 'mfcc4_std',
                'duration']

coefs = np.abs(lr.coef_[0])
imp = sorted(zip(feature_names, coefs), key=lambda x: -x[1])[:10]
print('Top features:')
for name, coef in imp:
    print(f'  {name:15s} {coef:.4f}')

# Save results
final_results = {
    'dataset': 'StandUp4AI_subset',
    'n_samples': len(y),
    'n_videos': len(set(videos)),
    'positive_rate': float(y.mean()),
    'features': f'{X.shape[1]}-dim (F0 + spectral)',
    'models': {k: {'f1': v['f1'], 'std': v['std']} for k,v in results.items()},
    'standup4ai_baseline': 0.51,
    'top_features': imp[:5],
}
with open(BASE / 'standup4ai_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f'\n✅ Results saved to {BASE / "standup4ai_results.json"}')